# 03 · The Lab instruments – tracing and monitoring

You cannot improve what you cannot see. Foundry records what your AI app does as **OpenTelemetry traces** in the **Application Insights** that `azd up` connected to the project. In this chapter you will:

1. switch tracing on from Python
2. make some calls, including a custom span with your own attributes
3. look at them in **Application Insights** in the Azure portal (and later in an agent's **Traces** tab in Foundry)
4. query them with **KQL** in Log Analytics

> ⚠️ Prompt and response *content* may contain personal data. Content recording is switched on here for the workshop; leave it off in production unless you need it. See [docs/observability.md](../docs/observability.md).

**Restart the kernel before running this notebook** (the tracing flags must be set before the SDK is imported).

In [ ]:
import os
os.environ["AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING"] = "true"          # emit GenAI semantic-convention spans
os.environ["AZURE_TRACING_GEN_AI_CONTENT_RECORDING_ENABLED"] = "true"   # also record prompts/responses (workshop only)

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *

from azure.monitor.opentelemetry import configure_azure_monitor
from opentelemetry import trace

# The project knows its Application Insights resource (the connection created by Bicep).
conn = project().telemetry.get_application_insights_connection_string()
configure_azure_monitor(connection_string=conn)
tracer = trace.get_tracer("stark.workshop")
print("tracing enabled")

## Generate traffic
We wrap the work in our own span so the whole scenario shows up as one trace, tagged with attributes you can filter on.

In [ ]:
oai = openai_client()
MODEL = model("chat")

with tracer.start_as_current_span("workshop-scenario") as span:
    span.set_attribute("stark.hero", "tony-stark")
    span.set_attribute("stark.chapter", "03")

    conv = oai.conversations.create()
    span.set_attribute("stark.conversation_id", conv.id)
    for q in ["Give me a status report on the arc reactor.",
              "Summarize that in five words.",
              "Now in Latin."]:
        r = oai.responses.create(model=MODEL, conversation=conv.id, input=q)
        print(q, "->", r.output_text)

print("conversation id:", conv.id)

In [ ]:
# Make sure the exporter sends everything before we look for it.
trace.get_tracer_provider().force_flush()

## Look in the portal
**Application Insights** (Azure portal → resource group `rg-stark-<env>` → `appi-stark-<env>`, the name is in `STARK_APPINSIGHTS`) shows this traffic:

1. **Investigate → Transaction search**, last 30 minutes. Look for the `workshop-scenario` span and the model calls under it. Search for the conversation id printed above.
2. Open an item: **End-to-end transaction details** shows the span tree, durations and (because content recording is on) the messages.
3. Telemetry needs **2–5 minutes** to appear.

The **Foundry** portal's **Traces** view is a per-agent view (**Build → Agents →** an agent **→ Traces**). These calls have no agent, so they are not listed there. From chapter 04 on, an agent's calls show up in its **Traces** tab, with the conversation id.

## Look with KQL
Traces live in the Log Analytics tables behind Application Insights. The attributes follow the OpenTelemetry **GenAI semantic conventions** (`gen_ai.*`), so the same query works for any framework that follows them.

In [ ]:
q = '''
AppDependencies
| where TimeGenerated > ago(2h)
| extend op    = tostring(Properties["gen_ai.operation.name"]),
         model = tostring(Properties["gen_ai.request.model"]),
         tin   = toint(Properties["gen_ai.usage.input_tokens"]),
         tout  = toint(Properties["gen_ai.usage.output_tokens"])
| where isnotempty(op)
| project TimeGenerated, Name, op, model, DurationMs, tin, tout, OperationId
| order by TimeGenerated desc
'''
df = kql_wait(q, hours=2)
df.head(20)

### Tokens and latency per model
The two numbers that drive cost and user experience.

In [ ]:
if df is not None and len(df):
    summary = df.groupby("model").agg(calls=("Name", "count"), avg_ms=("DurationMs", "mean"),
                                       tokens_in=("tin", "sum"), tokens_out=("tout", "sum")).round(0)
    display(summary)

In [ ]:
q_custom = '''
AppDependencies
| where TimeGenerated > ago(2h)
| where Name == "workshop-scenario" or tostring(Properties["stark.hero"]) != ""
| project TimeGenerated, Name, hero = tostring(Properties["stark.hero"]), chapter = tostring(Properties["stark.chapter"]), DurationMs, OperationId
'''
kql_wait(q_custom, hours=2)

## Monitoring agents
When you create agents (chapter 04), the portal adds a **Monitor** tab per agent: run counts, success rate, latency, token usage and evaluation scores over time – fed by the same Application Insights. Chapter 11 turns that into *continuous evaluation*.

## Recap
| Question | Where to look |
|---|---|
| What happened in this conversation? | Application Insights → Transaction search (or KQL on `OperationId`); portal → agent → Traces once you use an agent (chapter 04 on) |
| How slow / expensive are my calls? | KQL on `AppDependencies` (`gen_ai.usage.*`, `DurationMs`) |
| Is my agent healthy over time? | Portal → agent → Monitor (from chapter 04 on) |
| Can I add business context? | Custom spans and attributes (`stark.*`) |

> 💡 Next: Act II – the suit. [04 · J.A.R.V.I.S., the first agent](04_jarvis_prompt_agent.ipynb).